# QAOA for triangle MaxCut

Compare the p=1 QAOA cost landscape for a three-node triangle graph.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

QAOA alternates cost and mixer layers. Here a parameter grid produces a MaxCut cost landscape.

In [2]:
cost = SparsePauliOp.from_list([
    ("III", 1.5), ("IZZ", -0.5), ("ZZI", -0.5), ("ZIZ", -0.5)
])
parameters = [(gamma, beta) for gamma in np.linspace(0.0, np.pi, 7) for beta in np.linspace(0.0, np.pi / 2, 5)]

def qaoa_circuit(gamma, beta):
    circuit = QuantumCircuit(3)
    circuit.h(range(3))
    for first, second in ((0, 1), (1, 2), (0, 2)):
        circuit.rzz(float(-gamma), first, second)
    for wire in range(3):
        circuit.rx(float(2 * beta), wire)
    return circuit

circuits = [qaoa_circuit(*values) for values in parameters]

def reference_costs():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, cost)]).result()[0].data.evs.item() for c in circuits])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_costs)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_costs():
    return np.asarray([estimator.run([(c, cost)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_costs)
error = max_abs_error(reference, candidate)
best_match = int(np.argmax(reference)) == int(np.argmax(candidate))
method, device = qiskit_selection(estimator)

## 4. Check correctness before discussing speed

Every landscape point must agree within the declared tolerance.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/11_qaoa_maxcut.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="QAOA cost landscape atol=3e-6",
    passed=error <= 3e-6 and best_match,
    exact_match=best_match,
    selected_method=method,
    selected_device=device,
    metrics={"max_cost_error": error, "reference_best": float(reference.max()), "mettleq_best": float(candidate.max()), "best_parameters": parameters[int(np.argmax(candidate))]},
)


Comparison summary
------------------
Correctness contract: PASS — QAOA cost landscape atol=3e-6
SDK reference median: 22.666 ms
MettleQ median:       162.675 ms
Timing interpretation: the SDK reference was 7.177x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: yes

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "QAOA cost landscape atol=3e-6", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"best_parameters": [0.5235987755982988, 0.39269908169872414], "max_cost_error": 9.802331946140441e-07, "mettleq_best": 1.9620184302330017, "reference_best": 1.9620190528383281}, "mettleq_median_ms": 162.67458299989812, "notebook": "qiskit/11_qaoa_maxcut.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 22.666375007247552, "reference_over_mettleq": 0.13933568839860955, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


## What should you conclude?

Use MettleQ for larger statevector sweeps after profiling; this tiny landscape is a correctness lesson.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.